# Annotation Layers

Annotation layers are **reactive visual overlays** rendered on top of existing plots.
A layer is a function that receives either a time range or the data from a graph
on the plot, and returns a list of annotations (markers, spans, horizontal lines).
Layers re-evaluate automatically when the input changes.

Layers support **knobs** (tunable parameters) just like virtual products.

<div align="center">
<img src="https://github.com/SciQLop/SciQLop/raw/main/SciQLop/resources/icons/SciQLop.png"/>
</div>

## 1. Annotation types

Layers return lists of three annotation types:

| Type | Description | Key fields |
|------|-------------|------------|
| `Marker` | A point on the plot | `time`, `value`, `label`, `color` |
| `Span` | A vertical time interval | `start`, `stop`, `label`, `color` |
| `HLine` | A horizontal reference line | `value`, `label`, `color` |

In [ ]:
from SciQLop.user_api.plot import Marker, Span, HLine

# Examples of annotation objects
m = Marker(time=1e9, value=5.0, label="peak", color="#e74c3c")
s = Span(start=1e9, stop=2e9, label="interval", color="#3498db")
h = HLine(value=0.0, label="baseline", color="#2ecc71")

print(m, s, h)

## 2. The `%%layer` cell magic — a structural example

The easiest way to create a layer from a notebook. Define a function that takes
`(start, stop)` and returns a list of annotations. The layer appears in the
product tree under **Layers/** — drag it onto any plot.

This first example is purely structural: it doesn't read any data, just lays
down a regular grid of half-hour reference spans across the visible window.

In [ ]:
%%layer --path "examples/hour_grid"
import numpy as np

def hour_grid(start: float, stop: float) -> list[Span]:
    """Drop a half-hour reference span every hour across the visible range."""
    hour = 3600.0
    t = np.arange(start, stop, hour)
    return [Span(start=t0, stop=t0 + hour / 2, label=f"hour {i}",
                 color="#bdc3c7")
            for i, t0 in enumerate(t)]

The `Marker`, `Span`, and `HLine` names are automatically available inside `%%layer` cells.

Re-executing the cell **hot-reloads** the layer — any plot showing it will pick up
the new function on the next time-range change.

## 3. Data-aware layers — read from the plot

Most interesting layers compute annotations from the **actual data** shown on the plot.
Type-hint the `data` parameter with `Scalar`, `Vector`, `MultiComponent`, or
`Spectrogram` and SciQLop hands the layer a typed container with the matching graph's
data.

| Type hint | Matches |
|-----------|---------|
| `Scalar` | Line graph with 1 component |
| `Vector` | Line graph with 3+ components |
| `MultiComponent` | Line graph with >1 component |
| `Spectrogram` | Color map |
| *(no hint)* | First available graph |

The container exposes:
- `data.time` — 1D numpy array of timestamps
- `data.values` — 2D numpy array of shape `(N, ncomponents)`

Data-aware layers are re-evaluated when the **data changes**, not when the time
range changes.

The example below marks the highest-|B| sample on an MMS FGM panel — the marker
moves automatically when you pan to a new interval and fresh data arrives.

In [ ]:
import numpy as np
from SciQLop.user_api.plot import create_plot_panel, Vector, Marker
from SciQLop.user_api import TimeRange

p = create_plot_panel()
p.time_range = TimeRange("2015-11-18T02:14:30", "2015-11-18T03:34:00")
p.plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")

def b_max_marker(data: Vector) -> list[Marker]:
    """Mark the time of the largest |B| sample currently on screen."""
    mag = data.values[:, -1]  # MMS FGM last column is |B|
    if mag.size == 0:
        return []
    i = int(np.nanargmax(mag))
    return [Marker(time=data.time[i], value=mag[i],
                   label=f"|B|max = {mag[i]:.1f} nT", color="#e74c3c")]

p.add_layer(b_max_marker, plot_index=0)

## 4. Layers with knobs

Extra keyword arguments with defaults become tunable knobs, exactly like virtual products.
The knobs appear in the inspector panel when the layer is attached to a plot.

Use `Knob(widget="hline")` to make a threshold parameter appear as a **draggable
horizontal line** on the plot — much more intuitive than a spinbox for threshold tuning.

The example below finds the times when |B| crosses a user-controlled threshold.
Drag the red horizontal line to retune live.

In [ ]:
%%layer --path "examples/b_threshold_crossings"
import numpy as np
from typing import Annotated
from SciQLop.user_api.knobs import Knob
from SciQLop.user_api.layers import Vector

def b_threshold_crossings(
    data: Vector,
    threshold: Annotated[float, Knob(widget="hline", min=0.0, max=80.0,
                                     step=0.5, color="#e74c3c")] = 30.0,
) -> list[Marker]:
    """Mark every time |B| crosses the threshold.
    Drag the red horizontal line to change the threshold live."""
    mag = data.values[:, -1]  # |B|
    if mag.size < 2:
        return []
    crossings = np.where(np.diff(np.sign(mag - threshold)))[0]
    return [Marker(time=data.time[i], value=threshold, label="cross",
                   color="#e74c3c") for i in crossings]

Drag the layer onto an MMS FGM plot (or any panel showing a vector with |B|
as the last component) and the threshold-crossing markers update live as you
drag the red line up and down.

## 5. Programmatic attachment with `PlotPanel.add_layer()`

Instead of drag-and-drop, you can attach a layer directly to a plot from Python.
This example overlays the per-component min/max envelope on the FGM data:
horizontal lines for the extrema of each component (Bx, By, Bz).

In [ ]:
import numpy as np
from SciQLop.user_api.plot import create_plot_panel, Vector, HLine
from SciQLop.user_api import TimeRange

p = create_plot_panel()
p.time_range = TimeRange("2015-11-18T02:14:30", "2015-11-18T03:34:00")
p.plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")

COLORS = ["#3498db", "#27ae60", "#e74c3c"]
LABELS = ["Bx", "By", "Bz"]

def component_envelope(data: Vector) -> list[HLine]:
    """Draw min/max horizontal lines for each B-field component."""
    lines = []
    for j, (name, color) in enumerate(zip(LABELS, COLORS)):
        comp = data.values[:, j]
        if comp.size == 0:
            continue
        lines.append(HLine(value=float(np.nanmin(comp)),
                           label=f"min {name}", color=color))
        lines.append(HLine(value=float(np.nanmax(comp)),
                           label=f"max {name}", color=color))
    return lines

p.add_layer(component_envelope, plot_index=0)

The `plot_index` argument selects which subplot receives the layer (0 = first plot).
Extra keyword arguments — when the layer has knobs — set their initial values.

## 6. The `@register_layer` decorator

For scripts and plugins, use `@register_layer` to register a layer in the product tree
without the cell magic. The layer below highlights low-|B| intervals — useful for
spotting magnetic-field depressions or current sheet crossings.

In [ ]:
import numpy as np
from typing import Annotated
from SciQLop.user_api.layers import register_layer, Span, Vector
from SciQLop.user_api.knobs import Knob

@register_layer("examples/low_b_intervals")
def low_b_intervals(
    data: Vector,
    threshold: Annotated[float, Knob(widget="hline", min=0.0, max=50.0,
                                     step=0.5, color="#27ae60")] = 10.0,
    min_duration_s: float = 30.0,
) -> list[Span]:
    """Highlight contiguous intervals where |B| stays below `threshold`."""
    mag = data.values[:, -1]
    if mag.size < 2:
        return []
    below = mag < threshold
    edges = np.diff(below.astype(np.int8))
    starts = np.where(edges == 1)[0] + 1
    stops = np.where(edges == -1)[0] + 1
    if below[0]:
        starts = np.concatenate([[0], starts])
    if below[-1]:
        stops = np.concatenate([stops, [len(mag)]])
    return [Span(start=float(data.time[s]), stop=float(data.time[e - 1]),
                 color="#27ae60", label=f"|B|<{threshold:.0f}")
            for s, e in zip(starts, stops)
            if data.time[e - 1] - data.time[s] >= min_duration_s]

The layer is now available in the product tree under
**Layers/examples/low_b_intervals** — drag it onto an MMS FGM panel.

## 7. Layers with the fluent API

The fluent builder has a `.layer()` method that attaches a layer to the current subplot.
Here we mark the zero crossings of the Bz component — a common way to spot field
reversals across a current sheet.

In [ ]:
import numpy as np
from SciQLop.user_api.plot import fluent, Vector, Marker, HLine

def bz_zero_crossings(data: Vector) -> list[Marker | HLine]:
    """Mark every Bz zero-crossing and draw the zero reference line."""
    bz = data.values[:, 2]
    if bz.size < 2:
        return [HLine(value=0.0, color="#7f8c8d")]
    crossings = np.where(np.diff(np.sign(bz)))[0]
    return ([Marker(time=data.time[i], value=0.0, label="Bz=0",
                    color="#9b59b6") for i in crossings]
            + [HLine(value=0.0, color="#7f8c8d")])

panel = (fluent.new_panel()
    .plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")
    .layer(bz_zero_crossings)
    .time_range("2015-11-18T02:14:30", "2015-11-18T03:34:00"))

## 8. Mixing annotation types

A single layer can return a mix of markers, spans, and horizontal lines. The
example below annotates the visible interval with the mean |B| as a horizontal
reference line, the peak as a marker, and the longest above-mean stretch as a
shaded span.

In [ ]:
%%layer --path "examples/b_summary"
import numpy as np
from SciQLop.user_api.layers import Vector

def b_summary(data: Vector) -> list[Marker | Span | HLine]:
    mag = data.values[:, -1]
    t = data.time
    if mag.size == 0:
        return []
    mean = float(np.nanmean(mag))
    i_peak = int(np.nanargmax(mag))
    above = mag > mean
    edges = np.diff(above.astype(np.int8))
    starts = np.where(edges == 1)[0] + 1
    stops = np.where(edges == -1)[0] + 1
    if above[0]:
        starts = np.concatenate([[0], starts])
    if above[-1]:
        stops = np.concatenate([stops, [len(mag)]])
    span = []
    if len(starts) and len(stops):
        widths = t[stops - 1] - t[starts]
        k = int(np.argmax(widths))
        span = [Span(start=float(t[starts[k]]), stop=float(t[stops[k] - 1]),
                     label="longest above mean", color="#f39c12")]
    return span + [
        HLine(value=mean, label=f"mean = {mean:.1f} nT", color="#2ecc71"),
        Marker(time=t[i_peak], value=mag[i_peak],
               label=f"peak = {mag[i_peak]:.1f} nT", color="#e74c3c"),
    ]

## API reference

### Annotation types

```python
from SciQLop.user_api.plot import Marker, Span, HLine
# or
from SciQLop.user_api.layers import Marker, Span, HLine
```

| Type | Required fields | Optional fields |
|------|----------------|----------------|
| `Marker(time, value)` | `time: float`, `value: float` | `label`, `color`, `meta` |
| `Span(start, stop)` | `start: float`, `stop: float` | `label`, `color`, `meta` |
| `HLine(value)` | `value: float` | `label`, `color`, `meta` |

### Visual knobs in layers

Layer knobs support the same visual knob types as virtual products:

```python
from typing import Annotated
from SciQLop.user_api.knobs import Knob
from SciQLopPlots import SciQLopPlotRange

# Draggable horizontal line (threshold)
def my_layer(data: Scalar,
             threshold: Annotated[float, Knob(widget="hline")] = 5.0):
    ...

# Draggable vertical span (time window)
def my_layer(start, stop,
             window: SciQLopPlotRange = SciQLopPlotRange(0.3, 0.7)):
    ...
```

### Data type hints

```python
from SciQLop.user_api.plot import Scalar, Vector, MultiComponent, Spectrogram
# or
from SciQLop.user_api.layers import Scalar, Vector, MultiComponent, Spectrogram
```

These serve as both **type hints** and **data containers**. At runtime, the callback
receives an instance with `.time` (1D) and `.values` (2D) attributes.

### Registration

| Method | Use case |
|--------|----------|
| `%%layer` | Notebook cell magic |
| `@register_layer(path)` | Decorator for scripts and plugins |
| `panel.add_layer(func, plot_index=0, **knobs)` | Attach directly to a plot |
| `builder.layer(func, **knobs)` | Fluent API |

### Layer callback signatures

**Range-only** (re-evaluated on time-range change):
```python
def my_layer(start: float, stop: float, **knobs) -> list[Annotation]:
    ...
```

**Data-aware** (re-evaluated on data change):
```python
from SciQLop.user_api.plot import Vector

def my_layer(data: Vector, **knobs) -> list[Annotation]:
    t = data.time          # 1D numpy array of timestamps
    v = data.values        # 2D numpy array, shape (N, ncomponents)
    ...
```

Type-hint `data` with `Scalar`, `Vector`, `MultiComponent`, or `Spectrogram`
to filter which graph provides the data. Omit the hint to use the first available graph.